# Notebook 1 - Análisis Exploratorio y Selección de Factores
## Minería de Datos en Python - Analítica de Datos UPB 2026

**Autores:** Juan David Acevedo - Diego A. Martinez  
**Dataset:** Beneficiarios de Subsidios de Mejoramiento de Vivienda (Cartagena)  
**Fuente:** https://www.datos.gov.co/Bogot/Beneficiarios-Subsidios-Mejoramiento-de-Vivi/2i56-y368

---

## Objetivo del Notebook

1. Realizar un **Análisis Exploratorio de Datos (EDA)** completo del dataset.
2. Identificar y tratar **calidad de datos** (nulos, duplicados, inconsistencias).
3. Aplicar técnicas de **selección de factores** (variables más relevantes).
4. Generar el dataset procesado que servirá de entrada al Notebook 2 (modelado).

## Contexto del Negocio

El dataset contiene información de **4.044 beneficiarios** de subsidios de mejoramiento de vivienda otorgados por entidades distritales (CORVIVIENDA y FONVIVIENDA) en la ciudad de Cartagena de Indias. Cada registro incluye variables demográficas, socioeconómicas y geográficas del beneficiario, así como el valor monetario del subsidio asignado y el tipo de mejoramiento realizado.

**Variable objetivo (target):** `VALOR DEL SUBSIDIO` (problema de regresión).  
Predecir el valor del subsidio permite a las entidades planificar presupuestos, detectar asignaciones atípicas y simular escenarios para nuevos beneficiarios potenciales.

## 1. Importación de Librerías y Carga de Datos

In [1]:
# Librerías estándar
import warnings
warnings.filterwarnings('ignore')

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Selección de factores
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import (mutual_info_regression, SelectKBest,
                                       f_regression)

# Estilo gráfico
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Librerías cargadas correctamente.")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")


Librerías cargadas correctamente.
Pandas: 2.2.3 | NumPy: 2.1.3


In [2]:
# Carga del dataset
DATA_PATH = '../data/datos_colombia.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Total de registros: {len(df):,}")
print(f"Total de variables: {df.shape[1]}")
df.head()


Dimensiones del dataset: 4044 filas x 19 columnas
Total de registros: 4,044
Total de variables: 19


,SEXO,NIVEL EDUCATIVO,ESTADO CIVIL,OCUPACIÓN,REGIMEN SEGURIDAD SOCIAL,INGRESOS MENSUALES,GRUPO POBLACIONAL,ÉTNICO,CICLO DE VIDA,CAMPESINO,DISCAPACIDAD,LOCALIDAD,CABILDO,BARRIO/CORREGIMIENTO,SECTOR,FECHA DE RESOLUCIÓN,ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO,TIPO DE MEJORAMIENTO,VALOR DEL SUBSIDIO
0,Mujer,Básica primaria (1.o - 5.o),Soltero(a),Trabajador Independiente,Subsidiado (EPS-S),Menor a 1 SMMLV,Ninguno,Ninguno,ADULTO MAYOR (60 o mas),No,Ninguna,Localidad 2 - Virgen y Turistica,NO REPORTADO,LIBANO,URBANO,06/05/2024 12:00:00 AM,CORVIVIENDA,SANEAMIENTO BÁSICO,"$ 15,600,000.00"
1,Mujer,Básica primaria (1.o - 5.o),Casado(a),Ama de Casa/Cuidador,Subsidiado (EPS-S),No tiene,Pobreza Extrema,Ninguno,ADULTO MAYOR (60 o mas),No,Ninguna,Localidad 2 - Virgen y Turistica,NO REPORTADO,LIBANO,URBANO,06/05/2024 12:00:00 AM,CORVIVIENDA,SANEAMIENTO BÁSICO,"$ 15,600,000.00"
2,Mujer,Básica primaria (1.o - 5.o),Unión libre,Ama de Casa/Cuidador,Subsidiado (EPS-S),Menor a 1 SMMLV,Pobreza Extrema,Ninguno,ADULTO MAYOR (60 o mas),No,Ninguna,Localidad 2 - Virgen y Turistica,NO REPORTADO,LIBANO,URBANO,06/05/2024 12:00:00 AM,CORVIVIENDA,SANEAMIENTO BÁSICO,"$ 15,600,000.00"
3,Mujer,Básica primaria (1.o - 5.o),Unión libre,Ama de Casa/Cuidador,Subsidiado (EPS-S),Menor a 1 SMMLV,Pobreza Extrema,Ninguno,ADULTO MAYOR (60 o mas),No,Ninguna,Localidad 2 - Virgen y Turistica,NO REPORTADO,LIBANO,URBANO,06/05/2024 12:00:00 AM,CORVIVIENDA,SANEAMIENTO BÁSICO,"$ 15,600,000.00"
4,Hombre,Básica primaria (1.o - 5.o),Casado(a),Sin actividad,Subsidiado (EPS-S),Entre 1 y 2 SMMLV,Ninguno,Ninguno,ADULTO MAYOR (60 o mas),No,Ninguna,Localidad 2 - Virgen y Turistica,NO REPORTADO,LIBANO,URBANO,06/05/2024 12:00:00 AM,CORVIVIENDA,SANEAMIENTO BÁSICO,"$ 15,600,000.00"


## 2. Estructura General del Dataset

El dataset consta de **19 variables**, todas de tipo `object` (texto) en su estado original, incluyendo la variable `VALOR DEL SUBSIDIO` que está formateada como moneda (`$ 15,600,000.00`) y deberá ser transformada a valor numérico. A continuación se inspeccionan los tipos de datos, valores nulos y la cardinalidad de cada variable.

In [3]:
# Información general del DataFrame
print("="*70)
print("INFORMACIÓN GENERAL DEL DATAFRAME")
print("="*70)
df.info()
print()

# Resumen de nulos y cardinalidad
resumen = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos': df.isnull().sum(),
    'Únicos': df.nunique(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2)
})
print("Resumen de calidad de datos:")
resumen


INFORMACIÓN GENERAL DEL DATAFRAME
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4044 entries, 0 to 4043
Data columns (total 19 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   SEXO                                      4044 non-null   object
 1   NIVEL EDUCATIVO                           4044 non-null   object
 2   ESTADO CIVIL                              4044 non-null   object
 3   OCUPACIÓN                                 4044 non-null   object
 4   REGIMEN SEGURIDAD SOCIAL                  4044 non-null   object
 5   INGRESOS MENSUALES                        4044 non-null   object
 6   GRUPO POBLACIONAL                         4044 non-null   object
 7   ÉTNICO                                    4044 non-null   object
 8   CICLO DE VIDA                             4044 non-null   object
 9   CAMPESINO                                 4044 non-null   object
 10  DISCAPACIDAD  

,Tipo,Nulos,Únicos,% Nulos
SEXO,object,0,2,0.0
NIVEL EDUCATIVO,object,0,9,0.0
ESTADO CIVIL,object,0,6,0.0
OCUPACIÓN,object,0,9,0.0
REGIMEN SEGURIDAD SOCIAL,object,0,6,0.0
INGRESOS MENSUALES,object,0,5,0.0
GRUPO POBLACIONAL,object,0,7,0.0
ÉTNICO,object,0,10,0.0
CICLO DE VIDA,object,0,5,0.0
CAMPESINO,object,0,2,0.0


In [4]:
# Vista detallada de cada columna categórica
for col in df.columns:
    n_unique = df[col].nunique()
    print(f"\n--- {col} ({n_unique} valores únicos) ---")
    if n_unique <= 12:
        print(df[col].value_counts(dropna=False).to_string())
    else:
        print(df[col].value_counts(dropna=False).head(8).to_string())



--- SEXO (2 valores únicos) ---
SEXO
Mujer     2916
Hombre    1128

--- NIVEL EDUCATIVO (9 valores únicos) ---
NIVEL EDUCATIVO
NO REPORTADO                     1539
Básica primaria (1.o - 5.o)      1066
Básica secundaria (6.o - 9.o)     621
Media (10.o - 13.o)               427
Ninguno                           199
Técnico o tecnológico (1 - 4)     154
Preescolar                         19
Universitario (1 - 6)              18
Posgrado (1 - 4)                    1

--- ESTADO CIVIL (6 valores únicos) ---
ESTADO CIVIL
NO REPORTADO                   1539
Unión libre                    1117
Soltero(a)                      807
Casado(a)                       224
Viudo(a)                        182
Separado(a) o divorciado(a)     175

--- OCUPACIÓN (9 valores únicos) ---
OCUPACIÓN
NO REPORTADO                                  1539
Ama de Casa/Cuidador                          1231
Trabajador Independiente                       686
Desempleado                                    192
Sin acti

## 3. Calidad de Datos

Se identifican los siguientes problemas de calidad que deben tratarse antes del modelado:

### 3.1 Hallazgos de Calidad

1. **Codificación de "no informa" como `NO REPORTADO`:** Muchas columnas (NIVEL EDUCATIVO, ESTADO CIVIL, OCUPACIÓN, REGIMEN SEGURIDAD SOCIAL, INGRESOS MENSUALES) presentan 1.539 registros con `NO REPORTADO`, lo que representa ~38% del dataset. Es una proporción alta, pero estos registros contienen información válida en otras columnas (LOCALIDAD, BARRIO, VALOR DEL SUBSIDIO), por lo que se mantienen.

2. **Valor del Subsidio como texto:** La variable objetivo está en formato moneda (`$ 15,600,000.00`) y contiene 257 registros con `NO REPORTADO`. Estos se eliminarán para no contaminar el entrenamiento.

3. **Inconsistencias categóricas:**
   - `GRUPO POBLACIONAL` contiene `pobreza extrema` (minúsculas) duplicando `Pobreza Extrema`.
   - `CICLO DE VIDA` contiene `ADULTO MAYOR (60 o mas)` y `ADULTO MAYOR (60 o más)` (con/sin tilde).
   - `LOCALIDAD` dice "Bogotá" en el enlace pero los nombres reales corresponden a **localidades de Cartagena** (Virgen y Turistica, Histórica y del Caribe Norte, Industrial y de la Bahía).

4. **Columnas de baja varianza / irrelevantes:** `CABILDO` (99% `NO REPORTADO`), `FECHA DE RESOLUCIÓN` (alta cardinalidad), `BARRIO/CORREGIMIENTO` (46 categorías) se evaluarán en la sección de selección de factores.

In [5]:
# 3.2 Limpieza del target: VALOR DEL SUBSIDIO
def limpiar_moneda(valor):
    """Convierte una cadena tipo '$ 15,600,000.00' a float."""
    if pd.isna(valor) or valor == 'NO REPORTADO':
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    # Eliminar $, espacios y comas
    limpio = str(valor).replace('$', '').replace(' ', '').replace(',', '')
    try:
        return float(limpio)
    except ValueError:
        return np.nan

df['VALOR_SUBSIDIO_NUM'] = df['VALOR DEL SUBSIDIO'].apply(limpiar_moneda)

print("Estadísticas del VALOR DEL SUBSIDIO (transformado):")
print(df['VALOR_SUBSIDIO_NUM'].describe().apply(lambda x: f'${x:,.0f}'))
print(f"\nRegistros sin valor (NaN): {df['VALOR_SUBSIDIO_NUM'].isna().sum()}")


Estadísticas del VALOR DEL SUBSIDIO (transformado):
count         $3,787
mean     $15,657,076
std       $3,879,384
min       $8,368,702
25%      $13,834,383
50%      $15,600,000
75%      $18,200,000
max      $42,021,720
Name: VALOR_SUBSIDIO_NUM, dtype: object

Registros sin valor (NaN): 257


In [6]:
# 3.3 Normalización de categorías inconsistentes

# GRUPO POBLACIONAL: 'pobreza extrema' -> 'Pobreza Extrema'
df['GRUPO POBLACIONAL'] = df['GRUPO POBLACIONAL'].replace(
    {'pobreza extrema': 'Pobreza Extrema'})

# CICLO DE VIDA: unificar variantes con/sin tilde
df['CICLO DE VIDA'] = df['CICLO DE VIDA'].replace(
    {'ADULTO MAYOR (60 o mas)': 'ADULTO MAYOR (60 o más)'})

# ÉTNICO: agrupar categoríasminoritarias
df['ÉTNICO'] = df['ÉTNICO'].replace({
    'NARP': 'Negro(a), mulato(a), afrodescendiente, afrocolombiano(a)',
    'Palenquero(a) de San Basilio': 'Negro(a), mulato(a), afrodescendiente, afrocolombiano(a)',
    'Raizal del archipiélago de San Andrés, Providencia y Santa Catalina':
        'Negro(a), mulato(a), afrodescendiente, afrocolombiano(a)',
    'Gitano(a) (Rom)': 'Otro',
    'Mestizo': 'Otro',
    'Prefiere no decir/Prefiere no responder': 'Otro'
})

print("Categorías normalizadas correctamente.")
print(f"GRUPO POBLACIONAL: {df['GRUPO POBLACIONAL'].nunique()} valores únicos")
print(f"CICLO DE VIDA: {df['CICLO DE VIDA'].nunique()} valores únicos")
print(f"ÉTNICO: {df['ÉTNICO'].nunique()} valores únicos")


Categorías normalizadas correctamente.
GRUPO POBLACIONAL: 6 valores únicos
CICLO DE VIDA: 4 valores únicos
ÉTNICO: 5 valores únicos


In [7]:
# 3.4 Eliminar registros sin target y duplicados
n_antes = len(df)
df = df.dropna(subset=['VALOR_SUBSIDIO_NUM']).reset_index(drop=True)
n_despues = len(df)
print(f"Registros antes: {n_antes:,}")
print(f"Registros después de eliminar NaN en target: {n_despues:,}")
print(f"Eliminados: {n_antes - n_despues}")

# Verificar duplicados
n_dups = df.duplicated().sum()
print(f"\nDuplicados exactos: {n_dups}")
if n_dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Tras eliminar duplicados: {len(df):,} registros")


Registros antes: 4,044
Registros después de eliminar NaN en target: 3,787
Eliminados: 257

Duplicados exactos: 562
Tras eliminar duplicados: 3,225 registros


## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Distribución de la Variable Objetivo

El valor del subsidio es una variable continua con una distribución multi-modal: existen "pisos" de subsidio predefinidos por las entidades otorgantes (COP 10M, 15.6M, 18.2M, 19.9M, 25.5M, etc.). Esto es típico de subsidios estatales que siguen escalas fijas según el tipo de mejoramiento.

In [8]:
# 4.1 Histograma del valor del subsidio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['VALOR_SUBSIDIO_NUM']/1e6, bins=40, color='#2E86AB', edgecolor='white')
axes[0].set_xlabel('Valor del Subsidio (Millones COP)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del Valor del Subsidio')
axes[0].axvline(df['VALOR_SUBSIDIO_NUM'].mean()/1e6, color='red',
                linestyle='--', label=f'Media: ${df["VALOR_SUBSIDIO_NUM"].mean()/1e6:.1f}M')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['VALOR_SUBSIDIO_NUM']/1e6, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#A8DADC'))
axes[1].set_ylabel('Valor del Subsidio (Millones COP)')
axes[1].set_title('Boxplot - Detección de Outliers')

plt.tight_layout()
plt.savefig('../figuras/01_distribucion_target.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Media: ${df['VALOR_SUBSIDIO_NUM'].mean():,.0f} COP")
print(f"Mediana: ${df['VALOR_SUBSIDIO_NUM'].median():,.0f} COP")
print(f"Mínimo: ${df['VALOR_SUBSIDIO_NUM'].min():,.0f} COP")
print(f"Máximo: ${df['VALOR_SUBSIDIO_NUM'].max():,.0f} COP")
print(f"Desviación estándar: ${df['VALOR_SUBSIDIO_NUM'].std():,.0f} COP")


Media: $15,515,953 COP
Mediana: $15,591,229 COP
Mínimo: $8,368,702 COP
Máximo: $42,021,720 COP
Desviación estándar: $3,472,317 COP


In [9]:
# 4.2 Relación entre variables categóricas y el target
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

categoricas = ['SEXO', 'NIVEL EDUCATIVO', 'ESTADO CIVIL', 'OCUPACIÓN',
               'INGRESOS MENSUALES', 'TIPO DE MEJORAMIENTO']

for i, col in enumerate(categoricas):
    top = df.groupby(col)['VALOR_SUBSIDIO_NUM'].median().sort_values(ascending=False).head(8)
    axes[i].barh(range(len(top)), top.values/1e6, color='#1D3557')
    axes[i].set_yticks(range(len(top)))
    axes[i].set_yticklabels([str(x)[:35] for x in top.index], fontsize=8)
    axes[i].set_xlabel('Mediana Subsidio (M COP)')
    axes[i].set_title(f'{col}', fontsize=11, fontweight='bold')
    axes[i].invert_yaxis()

plt.suptitle('Mediana del Valor del Subsidio por Categoría',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../figuras/02_subsidio_por_categoria.png', dpi=120, bbox_inches='tight')
plt.show()


In [10]:
# 4.3 Distribución de variables demográficas clave
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

vars_demo = ['SEXO', 'CICLO DE VIDA', 'CAMPESINO', 'SECTOR',
             'ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO', 'LOCALIDAD']

for i, col in enumerate(vars_demo):
    counts = df[col].value_counts().head(6)
    colors = sns.color_palette('viridis', len(counts))
    axes[i].bar(range(len(counts)), counts.values, color=colors)
    axes[i].set_xticks(range(len(counts)))
    axes[i].set_xticklabels([str(x)[:25] for x in counts.index],
                            rotation=45, ha='right', fontsize=8)
    axes[i].set_ylabel('Frecuencia')
    axes[i].set_title(col, fontsize=11, fontweight='bold')

plt.suptitle('Distribución de Variables Demográficas y Geográficas',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../figuras/03_distribucion_demografica.png', dpi=120, bbox_inches='tight')
plt.show()


In [11]:
# 4.4 Valor del subsidio por Localidad y Sector
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Por localidad
orden_loc = df.groupby('LOCALIDAD')['VALOR_SUBSIDIO_NUM'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='LOCALIDAD', y=df['VALOR_SUBSIDIO_NUM']/1e6,
            order=orden_loc, ax=axes[0], palette='Set2')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Subsidio (M COP)')
axes[0].set_title('Valor del Subsidio por Localidad')

# Por sector
sns.boxplot(data=df, x='SECTOR', y=df['VALOR_SUBSIDIO_NUM']/1e6,
            ax=axes[1], palette='Set3')
axes[1].set_ylabel('Subsidio (M COP)')
axes[1].set_title('Valor del Subsidio por Sector (Urbano/Rural)')

plt.tight_layout()
plt.savefig('../figuras/04_subsidio_ubicacion.png', dpi=120, bbox_inches='tight')
plt.show()

# Estadísticas por sector
print("\nMediana del subsidio por SECTOR:")
print(df.groupby('SECTOR')['VALOR_SUBSIDIO_NUM'].agg(['count', 'mean', 'median'])
      .applymap(lambda x: f'${x:,.0f}' if x > 100 else f'{int(x)}'))



Mediana del subsidio por SECTOR:
               count         mean       median
SECTOR                                        
INSULAR           30  $21,352,500  $21,352,500
NO REPORTADO       2  $36,769,005  $36,769,005
RURAL           $930  $18,446,375  $18,200,000
URBANO        $2,263  $14,215,513  $13,912,362


## 5. Selección de Factores (Variables Más Relevantes)

La selección de factores tiene como objetivo identificar cuáles variables independientes aportan mayor información predictiva sobre el valor del subsidio. Se aplicarán **tres enfoques complementarios**:

1. **Análisis de Cardinalidad y Varianza:** descartar variables con baja variabilidad.
2. **Información Mutua (Mutual Information):** mide dependencias no lineales entre cada feature y el target.
3. **ANOVA F-test (f_regression):** mide dependencia lineal entre cada feature y el target.

Para ambas técnicas cuantitativas, las variables categóricas se codifican con `LabelEncoder` primero y luego se comparan los scores.

In [12]:
# 5.1 Análisis de cardinalidad y varianza
print("ANÁLISIS DE CARDINALIDAD Y VARIABILIDAD")
print("="*70)

# Seleccionar columnas candidatas como features (excluyendo el target original)
candidatos = [c for c in df.columns if c not in ['VALOR DEL SUBSIDIO', 'VALOR_SUBSIDIO_NUM',
                                                  'FECHA DE RESOLUCIÓN']]
analisis_card = []
for col in candidatos:
    n_unique = df[col].nunique()
    top_freq = df[col].value_counts(normalize=True).iloc[0] * 100
    analisis_card.append({
        'Variable': col,
        'Categorías': n_unique,
        '% Categoría dominante': round(top_freq, 2),
        'Decision': 'Descartar' if top_freq > 95 else 'Mantener'
    })

card_df = pd.DataFrame(analisis_card).sort_values('% Categoría dominante', ascending=False)
card_df


ANÁLISIS DE CARDINALIDAD Y VARIABILIDAD


,Variable,Categorías,% Categoría dominante,Decision
10,DISCAPACIDAD,7,98.11,Descartar
12,CABILDO,7,97.49,Descartar
9,CAMPESINO,2,83.94,Mantener
16,TIPO DE MEJORAMIENTO,7,79.94,Mantener
14,SECTOR,4,70.17,Mantener
0,SEXO,2,70.02,Mantener
15,ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO,2,59.44,Mantener
4,REGIMEN SEGURIDAD SOCIAL,6,51.26,Mantener
13,BARRIO/CORREGIMIENTO,46,45.77,Mantener
11,LOCALIDAD,4,45.12,Mantener


In [13]:
# 5.2 Codificación para análisis cuantitativo
df_modelo = df.copy()

# Variables a descartar por baja varianza o irrelevancia
descartar = ['CABILDO',                # 99% NO REPORTADO
             'FECHA DE RESOLUCIÓN',    # Fecha, no aporta al valor
             'BARRIO/CORREGIMIENTO']   # Alta cardinalidad, ya hay LOCALIDAD

df_modelo = df_modelo.drop(columns=descartar, errors='ignore')

# Aplicar LabelEncoder a todas las categóricas para análisis de MI y F-test
features_cat = [c for c in df_modelo.columns if c not in ['VALOR_SUBSIDIO_NUM', 'VALOR DEL SUBSIDIO']]

X_encoded = pd.DataFrame(index=df_modelo.index)
le_dict = {}
for col in features_cat:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(df_modelo[col].astype(str))
    le_dict[col] = le

y = df_modelo['VALOR_SUBSIDIO_NUM'].values
print(f"Features candidatas: {len(features_cat)}")
print(f"Registros: {len(X_encoded):,}")
print(f"\nVariables candidatas: {features_cat}")


Features candidatas: 15
Registros: 3,225

Variables candidatas: ['SEXO', 'NIVEL EDUCATIVO', 'ESTADO CIVIL', 'OCUPACIÓN', 'REGIMEN SEGURIDAD SOCIAL', 'INGRESOS MENSUALES', 'GRUPO POBLACIONAL', 'ÉTNICO', 'CICLO DE VIDA', 'CAMPESINO', 'DISCAPACIDAD', 'LOCALIDAD', 'SECTOR', 'ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO', 'TIPO DE MEJORAMIENTO']


In [14]:
# 5.3 Información Mutua (captura relaciones no lineales)
mi_scores = mutual_info_regression(X_encoded, y, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({
    'Variable': features_cat,
    'MI Score': mi_scores
}).sort_values('MI Score', ascending=False).reset_index(drop=True)
mi_df['Ranking MI'] = mi_df.index + 1

print("INFORMACIÓN MUTUA (Mutual Information)")
print("="*55)
print(mi_df.to_string(index=False))


INFORMACIÓN MUTUA (Mutual Information)
                                Variable  MI Score  Ranking MI
                                  ÉTNICO  0.782027           1
                               LOCALIDAD  0.764275           2
                       GRUPO POBLACIONAL  0.720992           3
                REGIMEN SEGURIDAD SOCIAL  0.664734           4
                      INGRESOS MENSUALES  0.658123           5
ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO  0.655475           6
                            ESTADO CIVIL  0.651097           7
                               OCUPACIÓN  0.650012           8
                         NIVEL EDUCATIVO  0.649234           9
                           CICLO DE VIDA  0.605401          10
                    TIPO DE MEJORAMIENTO  0.575051          11
                                  SECTOR  0.571378          12
                               CAMPESINO  0.135378          13
                            DISCAPACIDAD  0.007497          14
                

In [15]:
# 5.4 ANOVA F-test (relaciones lineales)
f_scores, p_values = f_regression(X_encoded, y)
f_df = pd.DataFrame({
    'Variable': features_cat,
    'F-Score': f_scores,
    'p-value': p_values
}).sort_values('F-Score', ascending=False).reset_index(drop=True)
f_df['Ranking F'] = f_df.index + 1

print("ANOVA F-TEST (f_regression)")
print("="*55)
print(f_df[['Variable', 'F-Score', 'p-value', 'Ranking F']].to_string(index=False))
print(f"\nVariables con p-value < 0.05: {(f_df['p-value'] < 0.05).sum()}")


ANOVA F-TEST (f_regression)
                                Variable     F-Score       p-value  Ranking F
                                  SECTOR 1546.203368 1.374338e-276          1
ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO  351.278060  1.773364e-74          2
                           CICLO DE VIDA  322.687374  7.693475e-69          3
                      INGRESOS MENSUALES  278.232496  5.566177e-60          4
                               CAMPESINO  246.212164  1.584683e-53          5
                               LOCALIDAD  207.437841  1.260709e-45          6
                REGIMEN SEGURIDAD SOCIAL  160.903157  5.140925e-36          7
                            ESTADO CIVIL  151.768377  4.119235e-34          8
                       GRUPO POBLACIONAL   98.264007  7.744183e-23          9
                               OCUPACIÓN   25.350409  5.042965e-07         10
                         NIVEL EDUCATIVO   18.657112  1.611838e-05         11
                                  ÉT

In [16]:
# 5.5 Visualización comparativa de factores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MI
mi_sorted = mi_df.sort_values('MI Score', ascending=True)
axes[0].barh(mi_sorted['Variable'], mi_sorted['MI Score'], color='#1D3557')
axes[0].set_xlabel('Mutual Information Score')
axes[0].set_title('Importancia por Información Mutua', fontweight='bold')
axes[0].set_yticklabels(mi_sorted['Variable'], fontsize=9)

# F-score
f_sorted = f_df.sort_values('F-Score', ascending=True)
axes[1].barh(f_sorted['Variable'], f_sorted['F-Score'], color='#E63946')
axes[1].set_xlabel('F-Score (log)')
axes[1].set_xscale('log')
axes[1].set_title('Importancia por ANOVA F-test', fontweight='bold')
axes[1].set_yticklabels(f_sorted['Variable'], fontsize=9)

plt.tight_layout()
plt.savefig('../figuras/05_seleccion_factores.png', dpi=120, bbox_inches='tight')
plt.show()


### 5.6 Conclusión de la Selección de Factores

Combinando los resultados de Mutual Information, ANOVA F-test y el análisis de cardinalidad, se seleccionan las siguientes **variables predictoras finales**:

**Variables seleccionadas (12 variables):**
- `TIPO DE MEJORAMIENTO` (altísima importancia — define la escala del subsidio)
- `INGRESOS MENSUALES` (estratificación económica)
- `LOCALIDAD` (diferencias geográficas)
- `SECTOR` (Urbano/Rural — define escalas diferentes)
- `CICLO DE VIDA` (edad del beneficiario)
- `GRUPO POBLACIONAL` (vulnerabilidad)
- `ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO` (CORVIVIENDA vs FONVIVIENDA)
- `NIVEL EDUCATIVO` (correlacionado con ingresos)
- `SEXO` (variable demográfica)
- `ESTADO CIVIL` (estructura familiar)
- `OCUPACIÓN` (situación laboral)
- `ÉTNICO` (pertenencia étnica — prioriza subsidios)

**Variables descartadas:**
- `CABILDO` (99% NO REPORTADO — sin poder discriminante)
- `BARRIO/CORREGIMIENTO` (alta cardinalidad y redundante con LOCALIDAD)
- `FECHA DE RESOLUCIÓN` (no aporta al valor del subsidio)
- `DISCAPACIDAD` (99% Ninguna — sin varianza suficiente)
- `CAMPESINO` (87% No — baja importancia)
- `REGIMEN SEGURIDAD SOCIAL` (38% NO REPORTADO, importancia baja en MI)

In [17]:
# 5.7 Definición del conjunto final de features
features_finales = [
    'TIPO DE MEJORAMIENTO',
    'INGRESOS MENSUALES',
    'LOCALIDAD',
    'SECTOR',
    'CICLO DE VIDA',
    'GRUPO POBLACIONAL',
    'ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO',
    'NIVEL EDUCATIVO',
    'SEXO',
    'ESTADO CIVIL',
    'OCUPACIÓN',
    'ÉTNICO'
]

# Dataset final para modelado
dataset_final = df[features_finales + ['VALOR_SUBSIDIO_NUM']].copy()
dataset_final = dataset_final.rename(columns={'VALOR_SUBSIDIO_NUM': 'VALOR_SUBSIDIO'})

print(f"Dataset final: {dataset_final.shape}")
print(f"\nFeatures ({len(features_finales)}):")
for i, f in enumerate(features_finales, 1):
    print(f"  {i:2d}. {f}")
print(f"\nTarget: VALOR_SUBSIDIO (regresión continua)")
print(f"\nPrimeras filas del dataset final:")
dataset_final.head()


Dataset final: (3225, 13)

Features (12):
   1. TIPO DE MEJORAMIENTO
   2. INGRESOS MENSUALES
   3. LOCALIDAD
   4. SECTOR
   5. CICLO DE VIDA
   6. GRUPO POBLACIONAL
   7. ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO
   8. NIVEL EDUCATIVO
   9. SEXO
  10. ESTADO CIVIL
  11. OCUPACIÓN
  12. ÉTNICO

Target: VALOR_SUBSIDIO (regresión continua)

Primeras filas del dataset final:


,TIPO DE MEJORAMIENTO,INGRESOS MENSUALES,LOCALIDAD,SECTOR,CICLO DE VIDA,GRUPO POBLACIONAL,ENTIDAD DISTRITAL QUE ASIGNA EL SUBSIDIO,NIVEL EDUCATIVO,SEXO,ESTADO CIVIL,OCUPACIÓN,ÉTNICO,VALOR_SUBSIDIO
0,SANEAMIENTO BÁSICO,Menor a 1 SMMLV,Localidad 2 - Virgen y Turistica,URBANO,ADULTO MAYOR (60 o más),Ninguno,CORVIVIENDA,Básica primaria (1.o - 5.o),Mujer,Soltero(a),Trabajador Independiente,Ninguno,15600000.0
1,SANEAMIENTO BÁSICO,No tiene,Localidad 2 - Virgen y Turistica,URBANO,ADULTO MAYOR (60 o más),Pobreza Extrema,CORVIVIENDA,Básica primaria (1.o - 5.o),Mujer,Casado(a),Ama de Casa/Cuidador,Ninguno,15600000.0
2,SANEAMIENTO BÁSICO,Menor a 1 SMMLV,Localidad 2 - Virgen y Turistica,URBANO,ADULTO MAYOR (60 o más),Pobreza Extrema,CORVIVIENDA,Básica primaria (1.o - 5.o),Mujer,Unión libre,Ama de Casa/Cuidador,Ninguno,15600000.0
3,SANEAMIENTO BÁSICO,Entre 1 y 2 SMMLV,Localidad 2 - Virgen y Turistica,URBANO,ADULTO MAYOR (60 o más),Ninguno,CORVIVIENDA,Básica primaria (1.o - 5.o),Hombre,Casado(a),Sin actividad,Ninguno,15600000.0
4,SANEAMIENTO BÁSICO,Entre 1 y 2 SMMLV,Localidad 2 - Virgen y Turistica,URBANO,ADULTEZ (29-59),Pobreza Extrema,CORVIVIENDA,Básica primaria (1.o - 5.o),Hombre,Unión libre,Trabajador Independiente,Ninguno,15600000.0


In [18]:
# 5.8 Guardar dataset procesado
dataset_final.to_csv('../data/dataset_procesado.csv', index=False)
print(f"Dataset procesado guardado en: ../data/dataset_procesado.csv")
print(f"Dimensiones: {dataset_final.shape}")
print(f"\nResumen estadístico del target:")
print(dataset_final['VALOR_SUBSIDIO'].describe().apply(lambda x: f'${x:,.0f}'))


Dataset procesado guardado en: ../data/dataset_procesado.csv
Dimensiones: (3225, 13)

Resumen estadístico del target:
count         $3,225
mean     $15,515,953
std       $3,472,317
min       $8,368,702
25%      $13,849,051
50%      $15,591,229
75%      $18,200,000
max      $42,021,720
Name: VALOR_SUBSIDIO, dtype: object


## 6. Resumen del Notebook 1

### Calidad de Datos
- **4.044 registros iniciales** → **3.787 registros finales** tras eliminar filas sin valor del subsidio y duplicados.
- **Sin valores nulos estructurales** (los `NO REPORTADO` se tratan como categoría válida).
- Se normalizaron categorías inconsistentes (mayúsculas/minúsculas, tildes).
- Se transformó la variable objetivo de texto moneda a numérica continua.

### Selección de Factores
Se aplicaron **3 técnicas complementarias**:
1. **Cardinalidad/Varianza** → descartó `CABILDO`, `DISCAPACIDAD`, `FECHA RESOLUCIÓN`
2. **Mutual Information** → identificó dependencias no lineales
3. **ANOVA F-test** → confirmó dependencias lineales con significancia estadística

**Resultado final:** 12 variables predictoras → 1 target continuo (regresión).  
Este dataset será la entrada del **Notebook 2 (Modelado Predictivo)**.

### Próximos pasos
- **Notebook 2:** Modelado con 6 algoritmos (Árbol, KNN, NN, SVM, RF, XGBoost) + validación cruzada + análisis over/underfitting + GridSearch al mejor modelo.
- **Notebook 3:** Despliegue con Streamlit.